<a href="https://colab.research.google.com/github/hoanganh1105/scann-approximate-nearest-neighbor/blob/main/Minichatbot_6xScaNN_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cell 1**



In [ ]:
# @title 🛠️ 1. Cài đặt Thư viện (FINAL FIX: ScaNN + JAX + Numpy)
import os
import sys
import subprocess

print("📦 Đang dọn dẹp môi trường lần cuối...")

# 1. Gỡ bỏ sạch sẽ các thư viện đang đánh nhau
# (Gỡ cả jax và ml_dtypes để cài lại bản khớp nhau)
pkgs = ["numpy", "scann", "tensorflow", "ml_dtypes", "jax", "jaxlib", "sentence-transformers"]
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y"] + pkgs)

print("📦 Đang cài đặt lại theo thứ tự nghiêm ngặt...")

commands = [
    # 1. Cài ml_dtypes MỚI trước (Để thỏa mãn JAX)
    "pip install 'ml_dtypes>=0.5.0'",

    # 2. Cài Numpy CŨ (Để thỏa mãn ScaNN)
    "pip install 'numpy<2.0.0'",

    # 3. Cài ScaNN và Tensorflow
    "pip install scann==1.3.2 tensorflow",

    # 4. Cài các thư viện NLP còn lại
    "pip install sentence-transformers wikipedia-api neo4j google-generativeai langchain fastcoref accelerate"
]

for cmd in commands:
    print(f"   -> Running: {cmd}")
    subprocess.check_call(cmd, shell=True)

# Config môi trường
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings("ignore")

print("\n✅ ĐÃ SỬA XONG MỌI XUNG ĐỘT.")
print("🛑 BẮT BUỘC: Vào Menu 'Runtime' -> 'Restart Session' (Khởi động lại phiên) NGAY BÂY GIỜ!")

📦 Đang dọn dẹp môi trường lần cuối...
📦 Đang cài đặt lại theo thứ tự nghiêm ngặt...
   -> Running: pip install 'ml_dtypes>=0.5.0'
   -> Running: pip install 'numpy<2.0.0'
   -> Running: pip install scann==1.3.2 tensorflow
   -> Running: pip install sentence-transformers wikipedia-api neo4j google-generativeai langchain fastcoref accelerate

✅ ĐÃ SỬA XONG MỌI XUNG ĐỘT.
🛑 BẮT BUỘC: Vào Menu 'Runtime' -> 'Restart Session' (Khởi động lại phiên) NGAY BÂY GIỜ!


# **Cell 2**



In [ ]:
# @title 🔗 2. Cấu hình Kết nối (CHẠY LẠI CÁI NÀY)
import os
from google.colab import userdata
import google.generativeai as genai
from neo4j import GraphDatabase

try:
    # 1. Lấy Key từ Secrets
    GEMINI_KEY = userdata.get('GEMINI_API_KEY')
    NEO4J_URI = userdata.get('NEO4J_URI')
    NEO4J_USER = userdata.get('NEO4J_USER')
    NEO4J_PASS = userdata.get('NEO4J_PASS')

    # 2. Cấu hình Gemini
    genai.configure(api_key=GEMINI_KEY)
    llm_model = genai.GenerativeModel('models/gemini-2.5-flash')
    print("✅ Gemini API: OK")

    # 3. Cấu hình Neo4j Driver (Cái bạn đang thiếu)
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    driver.verify_connectivity()
    print("✅ Neo4j AuraDB: OK (Driver đã sẵn sàng)")

except Exception as e:
    print(f"❌ LỖI KẾT NỐI: {e}")
    print("👉 Hãy kiểm tra lại mục 'Secrets' (biểu tượng chìa khóa) bên trái màn hình.")

✅ Gemini API: OK
❌ LỖI KẾT NỐI: Failed to DNS resolve address 069bdb4d.databases.neo4j.io:7687: [Errno -2] Name or service not known
👉 Hãy kiểm tra lại mục 'Secrets' (biểu tượng chìa khóa) bên trái màn hình.


# **Cell 3**



In [ ]:
# @title 🤖 3. Tải Models (Chế độ Tự động: Hỗ trợ cả CPU & GPU)
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from fastcoref import FCoref
import torch
import os

# 1. Tự động phát hiện thiết bị
if torch.cuda.is_available():
    device = "cuda"
    print("🚀 Phát hiện GPU: Đang kích hoạt chế độ Tăng tốc.")
else:
    device = "cpu"
    print("🐢 Không có GPU: Đang chạy chế độ CPU (Sẽ chậm hơn ở khâu trích xuất).")

print("⏳ Đang tải models...")

# 2. Embedding Model (ScaNN/Vector Search)
# Lưu ý: 'device=device' để nó tự chuyển sang CPU nếu cần
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

# 3. REBEL Model (Extraction)
rebel_tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
# Nếu là CPU, load model ở chế độ float32 mặc định để tránh lỗi
rebel_model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large").to(device)

# 4. Coreference Model
# FCoref có thể chạy trên CPU nhưng cần chỉ định rõ
coref_model = FCoref(device=device)

# 5. Cross-Encoder (Reranking)
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)

print(f"✅ TẤT CẢ MODELS ĐÃ SẴN SÀNG TRÊN: {device.upper()}!")

🐢 Không có GPU: Đang chạy chế độ CPU (Sẽ chậm hơn ở khâu trích xuất).
⏳ Đang tải models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ TẤT CẢ MODELS ĐÃ SẴN SÀNG TRÊN: CPU!


# **Cell 4**



In [ ]:
# @title ⚙️ Helper & 1️⃣ GIAI ĐOẠN 1: ScaNN Chunking (An toàn, đã sửa lỗi)
import scann
import numpy as np
import re
from tqdm.notebook import tqdm
import wikipediaapi

# --- 0. SCANN HELPER (Đã Fix Crash) ---
def build_scann_index(vectors, num_neighbors=10):
    if vectors is None or len(vectors) == 0: return None
    n_data = len(vectors)
    norms = np.linalg.norm(vectors, axis=1)[:, np.newaxis]
    vectors = vectors / (norms + 1e-9)
    builder = scann.scann_ops_pybind.builder(vectors, num_neighbors, "dot_product")
    if n_data < 200:
        return builder.score_brute_force().build()
    else:
        n_leaves = int(np.sqrt(n_data))
        return builder.tree(num_leaves=n_leaves, num_leaves_to_search=max(1, n_leaves//2), training_sample_size=n_data) \
            .score_ah(2, anisotropic_quantization_threshold=0.2).reorder(num_neighbors).build()

# --- 1. CHUNKING LOGIC ---
def scann_semantic_chunking(text, model, threshold=0.5):
    sentences = re.split(r'(?<=[.?!])\s+', text)
    if not sentences: return []
    embeddings = model.encode(sentences, show_progress_bar=False)
    searcher = build_scann_index(embeddings, num_neighbors=5)
    final_chunks = []
    current_group = [sentences[0]]
    current_vec = embeddings[0]

    for i in range(1, len(sentences)):
        sim_score = np.dot(current_vec, embeddings[i]) / (np.linalg.norm(current_vec) * np.linalg.norm(embeddings[i]) + 1e-9)
        if sim_score >= threshold:
            current_group.append(sentences[i])
            current_vec = (current_vec + embeddings[i]) / 2.0
        else:
            final_chunks.append(" ".join(current_group))
            current_group = [sentences[i]]
            current_vec = embeddings[i]

    if current_group: final_chunks.append(" ".join(current_group))
    return final_chunks

# --- MAIN LOOP ---
TOPICS = ["Leonardo da Vinci", "List of works by Leonardo da Vinci", "Mona Lisa", "The Last Supper (Leonardo da Vinci)", "Science and inventions of Leonardo da Vinci"]
wiki = wikipediaapi.Wikipedia(user_agent='GraphRAG/v2', language='en')
all_chunks = []
global_chunk_id = 0

print("🚀 Bắt đầu ScaNN Pipeline Giai đoạn 1...")
for topic in tqdm(TOPICS):
    page = wiki.page(topic)
    if page.exists():
        chunks = scann_semantic_chunking(page.text[:10000], embedding_model) # Lấy 10k ký tự
        for c in chunks:
            if len(c) > 30:
                # Đảm bảo có source_topic
                all_chunks.append({"chunk_id": f"chk_{global_chunk_id}", "source_topic": topic, "content": c, "type": "text"})
                global_chunk_id += 1

print(f"✅ Đã tạo {len(all_chunks)} chunks sử dụng ScaNN Logic.")

🚀 Bắt đầu ScaNN Pipeline Giai đoạn 1...


  0%|          | 0/5 [00:00<?, ?it/s]

✅ Đã tạo 180 chunks sử dụng ScaNN Logic.


In [ ]:
# @title ⚙️ FIX: Nâng cấp Google Generative AI SDK
print("📦 Đang nâng cấp google-generativeai...")
!pip install -U google-generativeai
print("✅ Nâng cấp hoàn tất!")

📦 Đang nâng cấp google-generativeai...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: google-generativeai
    Found existing installation: google-generativeai 0.8.5
    Uninstalling google-generativeai-0.8.5:
      Successfully uninstalled google-generativeai-0.8.5


✅ Nâng cấp hoàn tất!


# **Cell 5**



In [ ]:
# @title ⛏️ GIAI ĐOẠN 2: Trích xuất Graph (REBEL - Direct Generation)
# Code này sẽ chạy sau khi Cell 4 tạo ra all_chunks

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import sys

# Kiểm tra biến cần thiết
if 'rebel_model' not in globals():
    raise NameError("❌ LỖI: Rebel Model chưa được tải. Hãy chạy Cell 3 trước.")

# 1. Hàm parse kết quả
def extract_rebel_triplets(text):
    triplets = []
    text = text.strip()
    text = text.replace("<s>", "").replace("<pad>", "").replace("</s>", "")

    tokens = text.split()
    relation, subject, object_ = '', '', ''
    current = 'x'

    for token in tokens:
        if token == "<triplet>":
            current = 't'
            if relation != '': triplets.append({'subject': subject.strip(), 'relation': relation.strip(), 'object': object_.strip()})
            relation = ''; subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '': triplets.append({'subject': subject.strip(), 'relation': relation.strip(), 'object': object_.strip()})
            object_ = ''
        elif token == "<obj>":
            current = 'o'; relation = ''
        else:
            if current == 't': subject += ' ' + token
            elif current == 's': object_ += ' ' + token
            elif current == 'o': relation += ' ' + token

    if relation != '' and subject != '' and object_ != '':
        triplets.append({'subject': subject.strip(), 'relation': relation.strip(), 'object': object_.strip()})
    return triplets

BATCH_SIZE = 8  # Tăng lên 16 hoặc 32 nếu GPU của bạn mạnh (Colab Pro)
gen_kwargs_fast = {
    "max_length": 128,        # Giảm từ 256 xuống 128 (vì triple thường ngắn)
    "length_penalty": 0,
    "num_beams": 1,           # Quan trọng: Chuyển từ 3 xuống 1 (Greedy Search) - Nhanh gấp 3 lần
    "num_return_sequences": 1,
}

knowledge_graph_triples = []
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Bắt đầu trích xuất NHANH từ {len(all_chunks)} chunks...")

# Chia nhỏ all_chunks thành các batch
for i in tqdm(range(0, len(all_chunks), BATCH_SIZE)):
    batch_chunks = all_chunks[i : i + BATCH_SIZE]
    texts = [c['content'] for c in batch_chunks]

    try:
        # Tokenize cả batch
        inputs = rebel_tokenizer(texts, max_length=256, padding=True, truncation=True, return_tensors='pt').to(device)

        # Sinh văn bản hàng loạt
        with torch.no_grad(): # Tắt gradient để nhẹ máy
            generated_tokens = rebel_model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                **gen_kwargs_fast
            )

        # Giải mã và parse
        decoded_texts = rebel_tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

        for idx, decoded_text in enumerate(decoded_texts):
            extracted = extract_rebel_triplets(decoded_text)
            for t in extracted:
                t['source_chunk_id'] = batch_chunks[idx]['chunk_id']
                knowledge_graph_triples.append(t)

    except Exception as e:
        continue

print(f"\n✅ Hoàn thành! Trích xuất được {len(knowledge_graph_triples)} triples.")

🚀 Bắt đầu trích xuất NHANH từ 180 chunks...


  0%|          | 0/23 [00:00<?, ?it/s]


✅ Hoàn thành! Trích xuất được 240 triples.


# **Cell 6**



In [ ]:
# @title ⚙️ 6. GIAI ĐOẠN 2-3 & 4: ScaNN Cleaning, Prediction & Indexing
from tqdm.notebook import tqdm
import numpy as np
import re

# Kiểm tra đầu vào
if 'knowledge_graph_triples' not in globals() or len(knowledge_graph_triples) == 0:
    # Nếu Cell 5 trích xuất REBEL/Gemini lỗi 0 triple
    raise ValueError("❌ LỖI: knowledge_graph_triples rỗng! Hãy kiểm tra Cell 5.")

# --- GĐ 2: CLEANING (Gom nhóm thực thể trùng bằng ScaNN) ---
print("🔹 Giai đoạn 2: Bắt đầu ScaNN Entity Resolution...")

# Hàm kiểm tra số (được dùng trong logic gộp Entity)
def is_numeric(text): return bool(re.search(r'\d', text))

# Lấy danh sách thực thể từ dữ liệu thô (sau REBEL)
entities = list(set([t['subject'] for t in knowledge_graph_triples] + [t['object'] for t in knowledge_graph_triples]))
entity_vecs = embedding_model.encode(entities, show_progress_bar=False)

# Xây dựng Index ScaNN cho Entity
entity_searcher = build_scann_index(entity_vecs, num_neighbors=10)

canonical_map = {}
visited = set()
for i, ent in enumerate(tqdm(entities, desc="Cleaning Entities")):
    if i in visited: continue

    # ScaNN Search: Ngưỡng rất cao cho việc gộp tên (0.96)
    neighbors, dists = entity_searcher.search(entity_vecs[i], final_num_neighbors=10)

    cluster = []
    for idx, dist in zip(neighbors, dists):
        if dist > 0.96:
            cluster.append(idx)
            visited.add(idx)

    if cluster:
        cluster_names = [entities[ix] for ix in cluster]
        canonical = max(cluster_names, key=len)
        for name in cluster_names:
            canonical_map[name] = canonical

# Áp dụng Map
clean_triples = []
for t in knowledge_graph_triples:
    s_new = canonical_map.get(t['subject'], t['subject'])
    o_new = canonical_map.get(t['object'], t['object'])
    if s_new != o_new:
        clean_triples.append({"subject": s_new, "relation": t['relation'], "object": o_new, "source_chunk_id": t['source_chunk_id']})
print(f"   -> Graph làm sạch: {len(clean_triples)} triples.")


# --- GĐ 3: LINK PREDICTION (Dự đoán liên kết ẩn bằng ScaNN) ---
print("🔹 Giai đoạn 3: ScaNN Link Prediction...")
new_links = []
existing_pairs = set((t['subject'], t['object']) for t in clean_triples)

for i, vec in enumerate(tqdm(entity_vecs, desc="Predicting Links")):
    neighbors, dists = entity_searcher.search(vec, final_num_neighbors=5)

    for idx, dist in zip(neighbors, dists):
        if i == idx: continue

        # Vùng "Semantic Related" (0.85 < dist < 0.96)
        if 0.85 < dist < 0.96:
            e1, e2 = entities[i], entities[idx]

            if (e1, e2) not in existing_pairs and (e2, e1) not in existing_pairs:
                new_links.append({"subject": e1, "relation": "SCANN_SEMANTIC_LINK", "object": e2, "source_chunk_id": "scann_prediction"})
                existing_pairs.add((e1, e2))

clean_triples.extend(new_links)
knowledge_graph_triples = clean_triples # Cập nhật triples đã clean
print(f"   -> Đã thêm {len(new_links)} liên kết tiềm năng.")


# --- GĐ 4: INDEXING MAIN KNOWLEDGE BASE (Chạy sau khi clean_triples đã được định nghĩa) ---
print("🔨 Giai đoạn 4: Building Final ScaNN Index...")

# Lấy danh sách thực thể đã được làm sạch
final_entities = list(set([t['subject'] for t in clean_triples] + [t['object'] for t in clean_triples]))

# Xây dựng Corpus (Chunks + Entities)
corpus = [c['content'] for c in all_chunks] + final_entities
index_map = {}

# Map lại ID để truy xuất
for i, item in enumerate(corpus):
    if i < len(all_chunks): index_map[i] = {"type": "text", "content": item}
    else: index_map[i] = {"type": "entity", "content": item}

# Mã hóa toàn bộ corpus
all_vectors = embedding_model.encode(corpus, show_progress_bar=True)

# Build Final Hybrid Searcher
searcher = build_scann_index(all_vectors, num_neighbors=20)

print("✅ Indexing hoàn tất! Hệ thống Full ScaNN đã sẵn sàng.")

🔹 Giai đoạn 2: Bắt đầu ScaNN Entity Resolution...


Cleaning Entities:   0%|          | 0/278 [00:00<?, ?it/s]

   -> Graph làm sạch: 201 triples.
🔹 Giai đoạn 3: ScaNN Link Prediction...


Predicting Links:   0%|          | 0/278 [00:00<?, ?it/s]

   -> Đã thêm 0 liên kết tiềm năng.
🔨 Giai đoạn 4: Building Final ScaNN Index...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Indexing hoàn tất! Hệ thống Full ScaNN đã sẵn sàng.


In [ ]:
# @title ⚙️ Check Model Status
if 'llm_model' in globals():
    print(f"Model LLM đang được sử dụng: {llm_model.model_name}")
else:
    print("❌ Lỗi: Biến 'llm_model' chưa được định nghĩa. Hãy chạy lại Cell 2.")

Model LLM đang được sử dụng: models/gemini-2.5-flash


# **Cell 7**

In [ ]:
# @title ⚙️ 7. GIAI ĐOẠN 5 & 6: Retrieval & Smart Semantic Cache (FINAL)
from sentence_transformers import util
import numpy as np

# --- Memory Setup (GĐ 6) ---
# Cần lưu thêm 'queries' (text gốc) để Cross-Encoder so sánh
memory_data = {"vectors": [], "responses": [], "queries": []}
memory_searcher = None

def update_memory_scann(query_text, query_vec, response_text):
    """Lưu câu hỏi, vector và câu trả lời vào bộ nhớ."""
    global memory_searcher
    memory_data["vectors"].append(query_vec)
    memory_data["responses"].append(response_text)
    memory_data["queries"].append(query_text) # Lưu text gốc

    # Re-build index
    if len(memory_data["vectors"]) > 0:
        vecs = np.array(memory_data["vectors"])
        memory_searcher = build_scann_index(vecs, num_neighbors=1)

def check_smart_cache(new_query, new_query_vec, verbose=False):
    """Giai đoạn 6: Kiểm tra bộ nhớ với cơ chế 2 lớp (Vector + Cross-Encoder)."""
    if not memory_searcher: return None

    # LỚP 1: ScaNN (Lọc thô - Nhanh)
    idx, dists = memory_searcher.search(new_query_vec, final_num_neighbors=1)
    similarity = dists[0]
    best_idx = idx[0]

    # Nếu vector không giống lắm (>0.85), bỏ qua luôn
    if similarity < 0.85:
        return None

    # LỚP 2: Cross-Encoder (Kiểm tra tinh - Chính xác)
    # Lấy câu hỏi cũ trong quá khứ ra
    cached_query_text = memory_data["queries"][best_idx]

    # Hỏi Cross-Encoder: "Hai câu này có giống nhau không?"
    # (Dùng model rerank đã load ở Cell 3)
    score = rerank_model.predict([new_query, cached_query_text])

    # In ra để debug xem nó thông minh cỡ nào
    if verbose:
        print(f"   🔍 Cache Check: '{new_query}' vs '{cached_query_text}'")
        print(f"      -> Vector Sim: {similarity:.4f} | Rerank Score: {score:.4f}")

    # Quyết định: Rerank Score thường là logit, nếu > 4 (hoặc > 0.9 tùy model) là trùng
    # Với model ms-marco, score > 2.0 là khá chắc chắn
    if score > 2.0:
        return memory_data["responses"][best_idx]

    return None

def retrieve_context_scann(query_vec, verbose=False):
    """Giai đoạn 5: Retrieval & Expansion."""
    # 1. Tìm kiếm ban đầu (K=50 để phủ rộng)
    idx, dists = searcher.search(query_vec, final_num_neighbors=50)

    context = []
    anchors = []
    text_found_count = 0

    for i in idx:
        item = index_map[i]
        if item['type'] == 'text' and text_found_count < 5:
            clean_content = item['content'].replace('\n', ' ').strip()
            context.append(f"[{item['type'].upper()}] {clean_content}")
            text_found_count += 1
        elif item['type'] == 'entity' and len(anchors) < 5:
            if item['content'] not in anchors: anchors.append(item['content'])

    # 2. Context Expansion
    if anchors:
        anchor_vec = embedding_model.encode(anchors[0])
        exp_idx, _ = searcher.search(anchor_vec, final_num_neighbors=5)
        for i in exp_idx:
            item = index_map[i]
            if item['type'] == 'text' and f"[TEXT] {item['content'].replace('\n', ' ').strip()}" not in context:
                clean_content = item['content'].replace('\n', ' ').strip()
                context.append(f"[Expanded-Rel] {clean_content}")

    return context

def chat_with_scann(query, verbose=False):
    """Hàm Chatbot chính."""
    query_vec = embedding_model.encode(query)

    # >>> Giai đoạn 6: SMART CACHE CHECK <<<
    cached_response = check_smart_cache(query, query_vec, verbose)
    if cached_response:
        if verbose: print("   ⚡ SMART CACHE HIT!")
        return f"⚡ (Smart Cache): {cached_response}"

    # >>> Giai đoạn 5: Retrieval <<<
    context_list = retrieve_context_scann(query_vec, verbose)

    if not context_list:
        return "Sorry, the system could not find any relevant information."

    # >>> Giai đoạn 7: Generation <<<
    full_context = "\n".join(context_list[:15]) # Lấy 15 dòng

    if verbose:
        print("   --- CONTEXT (LLM Input) ---")
        print(full_context[:500] + "...") # In một phần để check
        print("   ---------------------------")

    prompt = f"""
    You are an expert AI assistant focused on Leonardo da Vinci. Answer entirely in English.
    Use ALL relevant information from the context below to provide a COMPREHENSIVE and DETAILED answer.

    CONTEXT:
    {full_context}

    QUESTION: {query}
    DETAILED ANSWER:"""

    try:
        response = llm_model.generate_content(prompt, request_options={'timeout': 20})
        ans = response.text
        # Lưu vào bộ nhớ sau khi trả lời xong
        update_memory_scann(query, query_vec, ans)
        return ans
    except Exception as e:
        return f"API Error: {e}"

print("✅ Hệ thống Smart Cache (Vector + Cross-Encoder) đã kích hoạt!")

✅ Hệ thống Smart Cache (Vector + Cross-Encoder) đã kích hoạt!


# **Cell 8**

In [ ]:
# @title 🧪 9. Kiểm thử Chuyên sâu (Reset Cache Chuẩn & Run)
import time

if 'chat_with_scann' not in globals():
    raise NameError("❌ LỖI: Hàm 'chat_with_scann' chưa được định nghĩa. Hãy chạy Cell 7 trước.")

# --- BƯỚC KHẮC PHỤC: XÓA SẠCH BỘ NHỚ VỚI CẤU TRÚC CHUẨN ---
global memory_data, memory_searcher
# Thêm key "queries" để tương thích với Smart Cache
memory_data = {"vectors": [], "responses": [], "queries": []}
memory_searcher = None
print("⚡ Bộ nhớ ScaNN (Memory) đã được XÓA SẠCH (Cấu trúc mới gồm Queries).")
# -----------------------------------------------------------------

test_cases = [
    "Who is Leonardo da Vinci?",                  # Test #1
    "Where was Leonardo born?",                     # Test #2: Check Smart Cache (Né cache #1)
    "When was the Mona Lisa painted?",            # Test #3
    "Did Leonardo invent anything?",              # Test #4
    "Tell me about Leonardo's flying machine.",   # Test #5
    "List some famous paintings by Leonardo.",    # Test #6
    "Did he ever finish the Adoration of the Magi?",# Test #7
    "What is the date of his birth?",             # Test #8
    "Why is the Mona Lisa famous?",               # Test #9
    "Who is Leonardo da Vinci?"                   # Test #10: Check Smart Cache (Hit cache #1)
]

import time
from google.api_core import exceptions

# Hàm bổ trợ để gọi Gemini với cơ chế tự động đợi khi bị giới hạn (Retry Logic)
def safe_chat_with_scann(query, verbose=True):
    max_retries = 5
    retry_delay = 20  # Nếu bị lỗi 429, đợi 20 giây rồi thử lại

    for attempt in range(max_retries):
        try:
            # Gọi hàm chat gốc của bạn
            return chat_with_scann(query, verbose=verbose)
        except exceptions.ResourceExhausted:
            print(f"⚠️ Quá tải API (Rate Limit). Đang đợi {retry_delay}s để thử lại (Lần {attempt+1}/{max_retries})...")
            time.sleep(retry_delay)
        except Exception as e:
            # Nếu là lỗi khác (như lỗi mạng), cũng nên đợi một chút
            if "429" in str(e):
                print(f"⚠️ Lỗi 429 phát hiện. Đang nghỉ giải lao {retry_delay}s...")
                time.sleep(retry_delay)
                continue
            return f"❌ Lỗi hệ thống: {e}"
    return "❌ Thất bại sau nhiều lần thử lại do giới hạn API."

# --- CHẠY TEST CASES ---
print(f"🚀 BẮT ĐẦU CHẠY {len(test_cases)} TEST CASES (Đã kích hoạt cơ chế chống nghẽn API)\n")
print("="*60)

for i, query in enumerate(test_cases):
    print(f"\n❓ CÂU HỎI #{i+1}: {query}")
    print("-" * 30)

    start_t = time.time()

    # SỬ DỤNG HÀM SAFE_CHAT MỚI
    response = safe_chat_with_scann(query, verbose=True)

    print("-" * 30)
    print(f"🤖 Bot: {response}")
    print(f"⏱️ Thời gian: {time.time()-start_t:.2f}s")
    print("="*60)

    # Tăng sleep lên 6 giây để giữ tốc độ ~10 requests/phút (An toàn cho bản Free)
    time.sleep(6)

print("\n✅ BÀI KIỂM THỬ HOÀN TẤT.")

⚡ Bộ nhớ ScaNN (Memory) đã được XÓA SẠCH (Cấu trúc mới gồm Queries).
🚀 BẮT ĐẦU CHẠY 10 TEST CASES (Đã kích hoạt cơ chế chống nghẽn API)


❓ CÂU HỎI #1: Who is Leonardo da Vinci?
------------------------------
   --- CONTEXT (LLM Input) ---
[TEXT] Leonardo da Vinci (1452–1519) was an Italian polymath, regarded as the epitome of the "Renaissance Man", displaying skills in numerous diverse areas of study. While most famous for his paintings such as the Mona Lisa and the Last Supper, Leonardo is also renowned in the fields of civil engineering, chemistry, geology, geometry, hydrodynamics, mathematics, mechanical engineering, optics, physics, pyrotechnics, and zoology.
[TEXT] Biography Early life (1452–1472) Birth and background Leonar...
   ---------------------------
------------------------------
🤖 Bot: Leonardo di ser Piero da Vinci, commonly known as Leonardo da Vinci, was an Italian polymath of the High Renaissance (1452–1519). He is widely regarded as the epitome of the "Renaissance 

In [ ]:
# @title 🧪 9. Kiểm thử CHUYÊN SÂU (Offline Retrieval Test)
import time
import numpy as np

# --- 1. KIỂM TRA SỰ SẴN SÀNG ---
if 'retrieve_context_scann' not in globals():
    raise NameError("❌ LỖI: Hàm 'retrieve_context_scann' chưa được định nghĩa. Hãy chạy Cell 7 trước.")

# --- 2. HÀM TEST OFFLINE ---
def offline_retrieval_test(query):
    """Chỉ chạy Retrieval và Context Expansion, bỏ qua LLM và Cache."""
    # Khởi tạo vector query
    query_vec = embedding_model.encode(query)

    # Retrieval (Gọi trực tiếp Giai đoạn 5)
    context_list = retrieve_context_scann(query_vec, verbose=True)

    if not context_list or not "".join(context_list).strip():
        return "❌ Retrieval Failed: No relevant context found."

    # Format Context thành chuỗi để hiển thị
    full_context = "\n".join(context_list)

    # Trả về Context để người dùng tự đánh giá
    return full_context

# --- 3. DANH SÁCH CÂU HỎI TEST ---
test_cases = [
    "Who is Leonardo da Vinci?",                  # Test #1
    "Where was Leonardo born?",                     # Test #2
    "When was the Mona Lisa painted?",            # Test #3
    "Did Leonardo invent anything?",              # Test #4
    "Tell me about Leonardo's flying machine.",   # Test #5
    "List some famous paintings by Leonardo.",    # Test #6
    "Did he ever finish the Adoration of the Magi?",# Test #7
    "What is the date of his birth?",             # Test #8
    "Why is the Mona Lisa famous?",               # Test #9
    "What is Leonardo's full name?"               # Test #10: Ngữ nghĩa gần
]

print(f"🚀 BẮT ĐẦU CHẠY {len(test_cases)} TEST CASES (OFFLINE RETRIEVAL)\n")
print("="*60)

for i, query in enumerate(test_cases):
    print(f"\n❓ CÂU HỎI #{i+1}: {query}")
    print("-" * 30)

    start_t = time.time()

    # Gọi hàm test OFFLINE
    response = offline_retrieval_test(query)

    print("-" * 30)
    print("📦 CONTEXT (Tìm thấy):")
    # In toàn bộ Context tìm thấy
    print(response)
    print(f"\n⏱️ Thời gian: {time.time()-start_t:.2f}s (Chỉ tính Retrieval)")
    print("="*60)

    time.sleep(0.5)

print("\n✅ BÀI KIỂM THỬ HOÀN TẤT.")

🚀 BẮT ĐẦU CHẠY 10 TEST CASES (OFFLINE RETRIEVAL)


❓ CÂU HỎI #1: Who is Leonardo da Vinci?
------------------------------
------------------------------
📦 CONTEXT (Tìm thấy):
[TEXT] Leonardo da Vinci (1452–1519) was an Italian polymath, regarded as the epitome of the "Renaissance Man", displaying skills in numerous diverse areas of study. While most famous for his paintings such as the Mona Lisa and the Last Supper, Leonardo is also renowned in the fields of civil engineering, chemistry, geology, geometry, hydrodynamics, mathematics, mechanical engineering, optics, physics, pyrotechnics, and zoology.
[TEXT] Biography Early life (1452–1472) Birth and background Leonardo da Vinci, properly named Leonardo di ser Piero da Vinci ("Leonardo, son of ser Piero from Vinci"), was born on 15 April 1452 in, or close to, the Tuscan hill town of Vinci, Italy, 20 miles from Florence. He was born out of wedlock to Piero da Vinci (Ser Piero da Vinci d'Antonio di ser Piero di ser Guido; 1426–1504), a Fl

# **Cell 9**

In [ ]:
# @title 🚀 FINAL CHATBOT UI (Hợp nhất Logic & Giao diện)
from IPython.display import display, HTML, Javascript
import ipywidgets as widgets
import uuid, datetime, time
import numpy as np
from sentence_transformers import util
from google.colab import userdata
from google.api_core.exceptions import DeadlineExceeded

# --- KIỂM TRA BIẾN CỐT LÕI (SAFETY CHECK) ---
if 'driver' not in locals():
    print("⚠️ WARNING: Neo4j driver not defined. Chatbot will run in Text-Only mode.")
if 'embedding_model' not in locals():
    print("❌ ERROR: Please run Cell 3 (Load Models) first.")
    raise NameError("embedding_model is not defined.")


# ------------------- RAG BACKEND LOGIC (Giai đoạn 4, 5, 6) -------------------

# Khởi tạo bộ nhớ
if 'memory_store' not in locals():
    memory_store = {"vectors": [], "answers": []}

def retrieve_context(query, verbose=False):
    # Dùng k=30 (Lưới rộng)
    q_vec = embedding_model.encode(query)
    idx, dists = searcher.search(q_vec, final_num_neighbors=30)

    context = []
    anchors = []
    text_found_count = 0

    for i in idx:
        if i not in index_map: continue
        item = index_map[i]

        # 1. Text Chunk: Chỉ lấy 5 cái tốt nhất
        if item['type'] == 'text' and text_found_count < 5:
            context.append(f"[{item['type']}] {item['content']}")
            text_found_count += 1

        # 2. Entity: Lấy làm Anchor cho Graph
        elif item['type'] == 'entity':
            anchors.append(item['content'])

    # 3. Graph Expansion (Chỉ chạy nếu driver tồn tại)
    valid_anchors = list(set(anchors))[:5]
    if valid_anchors and 'driver' in globals() and driver:
        try:
            with driver.session() as session:
                q = """
                MATCH (e:Entity)-[r:REL]-(o)
                WHERE e.name IN $names
                RETURN e.name AS subj, r.type AS rel, o.name AS obj
                LIMIT 15
                """
                res = session.run(q, names=valid_anchors)
                for r in res:
                    context.append(f"[Graph Fact] {r['subj']} -> {r['rel']} -> {r['obj']}")
        except Exception as e:
            if verbose: print(f"   ⚠️ Graph Error: {e}")

    return "\n".join(context)

def chat_system(query):
    # Giai đoạn 6: Memory Check
    q_vec = embedding_model.encode(query)
    if len(memory_store["vectors"]) > 0:
        scores = util.cos_sim(q_vec, np.array(memory_store["vectors"]))[0]
        if max(scores) > 0.95:
            return f"⚡ (Cache): {memory_store['answers'][np.argmax(scores)]}"

    # Retrieval
    context = retrieve_context(query)

    if not context.strip():
        return "Sorry, the system could not find any relevant information in the knowledge base."

    # Generation Prompt (ÉP BUỘC TIẾNG ANH VÀ CHI TIẾT)
    prompt = f"""
    You are an expert AI assistant focused on Leonardo da Vinci.

    [IMPORTANT] Language Requirement: ANSWER ENTIRELY IN ENGLISH.
    Use ALL relevant information from the context to provide a DETAILED and COMPREHENSIVE response.
    If the context does not contain the answer, state that the information is not found in the context.

    CONTEXT:
    {context}

    QUESTION: {query}
    DETAILED ANSWER:"""

    try:
        response = llm_model.generate_content(prompt, request_options={'timeout': 20})
        ans_text = response.text

        # Lưu vào bộ nhớ
        memory_store["vectors"].append(q_vec)
        memory_store["answers"].append(ans_text)

        return ans_text
    except Exception as e: return f"API Error: {e}"


# ------------------- UI & CITATION LOGIC -------------------

# ------------------- CSS -------------------
display(HTML("""
<style>
    /* CSS đã được cung cấp ở trên */
</style>
"""))

# ------------------- WIDGET DEFINITION -------------------
chat_box = widgets.HTML("<div class='chat-area' id='chat-box'></div>")
text_input = widgets.Text(placeholder="Ask about Leonardo da Vinci...", layout=widgets.Layout(flex='1 0 auto'))
send_btn = widgets.Button(description="Send", button_style='primary', layout=widgets.Layout(width="90px"))

# ------------------- HELPER FUNCTIONS -------------------
def scroll_bottom():
    display(Javascript("document.getElementById('chat-box').scrollTop=document.getElementById('chat-box').scrollHeight;"))

def add_bubble(msg, user=True):
    """Thêm bong bóng chat và HTML tùy chỉnh."""
    role = "user" if user else "bot"
    js_code = f"""
        let box=document.getElementById('chat-box');
        let div = document.createElement('div');
        div.className='bubble {role}';
        div.innerHTML = `{msg}`;
        box.appendChild(div);
        box.scrollTop = box.scrollHeight;
    """
    display(Javascript(js_code))

def add_typing():
    """Thêm hiệu ứng đang gõ."""
    global typing_id
    typing_id = str(uuid.uuid4())
    js_code = f"""
        let box=document.getElementById('chat-box');
        let div=document.createElement('div');
        div.className='typing';
        div.id='{typing_id}';
        div.innerHTML='<div class="dot"></div><div class="dot"></div><div class="dot"></div>';
        box.appendChild(div);
        box.scrollTop = box.scrollHeight;
    """
    display(Javascript(js_code))

def remove_typing():
    """Xóa hiệu ứng đang gõ."""
    global typing_id
    if typing_id:
        js_code = f"""
            let t=document.getElementById('{typing_id}');
            if(t) t.remove();
        """
        display(Javascript(js_code))
        typing_id=None

def extract_and_format_sources(context_str):
    """Phân tích context và format thành dấu gạch đầu dòng HTML."""
    graph_facts = []
    text_chunks = []

    for line in context_str.split('\n'):
        if "[Graph Fact]" in line:
            cleaned_fact = line.replace("[Graph Fact]", "🕸️").strip()
            graph_facts.append(cleaned_fact)
        elif "[Text]" in line:
            cleaned_text = line.replace("[Text]", "📄").strip()
            text_chunks.append(cleaned_text[:100] + "...")

    if not graph_facts and not text_chunks: return ""

    output = "<hr style='border-top: 1px solid #ccc; margin: 5px 0;'><small><strong>Sources:</strong><ul>"

    for fact in graph_facts:
        output += f"<li>{fact}</li>"

    for chunk in text_chunks:
        output += f"<li>{chunk}</li>"

    output += "</ul></small>"
    return output

# ------------------- MAIN EVENT HANDLER (RAG) -------------------
def call_rag_pipeline_and_update(_):
    """Chạy toàn bộ RAG pipeline và cập nhật giao diện."""
    msg = text_input.value.strip()
    if not msg: return

    text_input.value=""
    add_bubble(msg, user=True)

    # 1. Kiểm tra bộ nhớ (Stage 6 - Quick Check)
    q_vec = embedding_model.encode(msg)
    if len(memory_store["vectors"]) > 0:
        scores = util.cos_sim(q_vec, np.array(memory_store["vectors"]))[0]
        if max(scores) > 0.95:
            final_answer = memory_store['answers'][np.argmax(scores)]

            remove_typing()
            final_html = f"<strong>⚡ (Cache Hit)</strong> {final_answer}"
            add_bubble(final_html, user=False)
            return

    # 2. Generation (G)
    add_typing()
    context_str = retrieve_context(msg, verbose=False)

    final_answer = chat_system(msg)

    # 3. Format Nguồn trích dẫn
    citations_html = extract_and_format_sources(context_str)

    # 4. Final UI Update
    final_html = f"<strong>{final_answer}</strong>{citations_html}"

    remove_typing()
    add_bubble(final_html, user=False)
    scroll_bottom()


# ------------------- EVENT ATTACHMENT -------------------
send_btn.on_click(call_rag_pipeline_and_update)
text_input.on_submit(call_rag_pipeline_and_update)


# ------------------- INITIAL DISPLAY -------------------
display(widgets.HTML("<div class='chat-header'>🤖 GraphRAG AI Chatbot (Leonardo da Vinci Expert) 🧠</div>"))
display(chat_box)
display(widgets.HBox([text_input, send_btn]))

add_bubble("Hello! I am your Leonardo da Vinci expert chatbot. How can I help you?", user=False)
scroll_bottom()

HTML(value="<div class='chat-header'>🤖 GraphRAG AI Chatbot (Leonardo da Vinci Expert) 🧠</div>")

HTML(value="<div class='chat-area' id='chat-box'></div>")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>